# Phase 3: Baseline Modelling
## Project: Kataoka Inc. Predictive Maintenance System

---

## Business Context

The goal of this phase is to establish a performance benchmark for
Remaining Useful Life (RUL) prediction before any feature engineering
or hyperparameter tuning is applied.

A baseline model answers one critical question:
**How well can we predict RUL using raw sensor values alone?**

This benchmark is essential because:
- It gives us a reference point to measure improvement against
- It tells us whether the raw signal carries any predictive value at all
- It prevents us from over-engineering features that may not help

---

## What This Phase Covers

- Load the processed RUL dataset (17 failure robots, 15,245 readings)
- Perform robot-level train/test split to prevent data leakage
- Encode categorical features
- Apply sample weighting to address class imbalance
- Train three baseline models on raw sensor features only:
  Random Forest, Gradient Boosting, Support Vector Regression
- Evaluate with MAE, RMSE, R² overall and segmented by risk zone
- Log all runs in MLflow with stage=baseline tag

---

## Critical Rules

- Split is by robot_id, never by row
- No engineered features in this phase, raw sensor values only
- No hyperparameter tuning, default parameters only
- Sample weighting applied to give Critical zone readings more influence
- Evaluation reported per risk zone, not just overall
- MLflow logging is mandatory for every run

---

## Section 1: Imports and Data Loading

In [2]:
# Imports libraries and sets up environment for RUL predictive maintenance analysis.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load RUL dataset 
# This dataset contains only the 17 confirmed failure robots.
# All 15,245 rows have valid RUL labels grounded in confirmed failure times.

rul_df = pd.read_csv(
    "../data/processed/rul_dataset.csv",
    parse_dates=["timestamp", "failure_time"]
)

print("RUL dataset loaded.")
print(f"  Shape   : {rul_df.shape}")
print(f"  Robots  : {rul_df['robot_id'].nunique()}")
print(f"  Columns : {rul_df.columns.tolist()}")

RUL dataset loaded.
  Shape   : (15245, 25)
  Robots  : 17
  Columns : ['reading_id', 'robot_id', 'timestamp', 'vibration_level', 'motor_temperature', 'torque_load', 'power_consumption', 'model_type', 'factory_location', 'operating_environment', 'cumulative_hours', 'failure_time', 'rul_hours_raw', 'rul_hours', 'is_failure_robot', 'label_status', 'total_maintenance_count', 'total_downtime_hours', 'avg_downtime_hours', 'repair_count', 'replacement_count', 'lubrication_count', 'calibration_count', 'inspection_count', 'health_risk_label']


## Section 2: Feature Selection for Baseline Model

For the baseline model we use raw sensor values only.
No engineered features, no rolling statistics, no trend calculations.

This is intentional. The baseline must reflect model performance
without any feature engineering so we have a honest benchmark
to compare against in Phase 5.

Raw features used:
- vibration_level
- motor_temperature
- torque_load
- power_consumption
- cumulative_hours
- model_type (encoded)
- factory_location (encoded)
- operating_environment (encoded)
- maintenance aggregate features

Target variable: rul_hours

In [3]:
# Define feature columns for baseline model.
# Raw sensor readings, cumulative hours, robot context, maintenance aggregates.
# No rolling windows, no trend features, no engineered signals.

RAW_FEATURES = [
    # Raw sensor readings
    "vibration_level",
    "motor_temperature",
    "torque_load",
    "power_consumption",

    # Operational context
    "cumulative_hours",

    # Maintenance aggregates (from Phase 2)
    "total_maintenance_count",
    "total_downtime_hours",
    "avg_downtime_hours",
    "repair_count",
    "replacement_count",
    "lubrication_count",
    "calibration_count",
    "inspection_count",

    # Categorical context (will be encoded below)
    "model_type",
    "factory_location",
    "operating_environment",
]

TARGET = "rul_hours"

print(f"Features selected : {len(RAW_FEATURES)}")
print(f"Target variable   : {TARGET}")
print(f"\nFeature list:")
for f in RAW_FEATURES:
    print(f"  {f}")

Features selected : 16
Target variable   : rul_hours

Feature list:
  vibration_level
  motor_temperature
  torque_load
  power_consumption
  cumulative_hours
  total_maintenance_count
  total_downtime_hours
  avg_downtime_hours
  repair_count
  replacement_count
  lubrication_count
  calibration_count
  inspection_count
  model_type
  factory_location
  operating_environment


## Section 3: Encode Categorical Features

Three categorical columns need to be converted to numbers
before the model can process them:
- model_type (5 categories)
- factory_location (5 categories)
- operating_environment (4 categories)

We use Label Encoding here for the baseline model.
This is simple and appropriate for tree-based models
which do not assume any ordinal relationship between categories.

In [4]:
# Encode categorical features
# We create a working copy to avoid modifying the original dataframe.
# Label encoding converts each unique category to an integer.
# Tree-based models (Random Forest, Gradient Boosting) handle this well.

model_df = rul_df.copy()

categorical_cols = ["model_type", "factory_location", "operating_environment"]
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    model_df[col + "_encoded"] = le.fit_transform(model_df[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Update RAW_FEATURES to use encoded versions
ENCODED_FEATURES = [
    f if f not in categorical_cols else f + "_encoded"
    for f in RAW_FEATURES
]

print(f"\nFinal feature list ({len(ENCODED_FEATURES)} features):")
for f in ENCODED_FEATURES:
    print(f"  {f}")

model_type: {'ARM-X7': np.int64(0), 'ARM-X9': np.int64(1), 'ASSEMBLY-A3': np.int64(2), 'CONVEYOR-C2': np.int64(3)}
factory_location: {'Fukuoka Center': np.int64(0), 'Hokkaido Lab': np.int64(1), 'Nagoya Site': np.int64(2), 'Osaka Plant': np.int64(3), 'Tokyo Factory': np.int64(4)}
operating_environment: {'Clean Room (20-22°C, 40% RH)': np.int64(0), 'Climate Controlled (22-25°C, 50% RH)': np.int64(1), 'Heavy Industrial (30-40°C, 70% RH)': np.int64(2), 'Standard Factory (25-30°C, 60% RH)': np.int64(3)}

Final feature list (16 features):
  vibration_level
  motor_temperature
  torque_load
  power_consumption
  cumulative_hours
  total_maintenance_count
  total_downtime_hours
  avg_downtime_hours
  repair_count
  replacement_count
  lubrication_count
  calibration_count
  inspection_count
  model_type_encoded
  factory_location_encoded
  operating_environment_encoded


## Section 4: Robot-Level Train/Test Split

We split by robot_id, not by row.

Splitting by row would allow the model to see readings from
the same robot in both training and test sets. Because sensor
readings from the same robot are temporally correlated, this
would constitute data leakage. The model would learn robot-specific
patterns rather than generalising across the fleet.

Splitting by robot ensures the test set contains robots the model
has never seen during training. This is the honest evaluation
that reflects real deployment conditions.

Split strategy:
- 13 robots for training
- 4 robots for testing
- Reviewed for data volume balance across robot types

In [5]:
# Robot-level train/test split
# We sort robots by their total reading count to ensure the test set
# does not accidentally contain only short-coverage robots.

robot_sizes = (
    model_df.groupby("robot_id")["rul_hours"]
    .count()
    .sort_values(ascending=False)
    .reset_index()
)
robot_sizes.columns = ["robot_id", "reading_count"]

# Merge model type for visibility
robot_sizes = robot_sizes.merge(
    rul_df[["robot_id", "model_type"]].drop_duplicates(),
    on="robot_id"
)

print("Robot Reading Counts (sorted)")
display(robot_sizes)

# Assign train/test robots
# We hold out 4 robots for testing, selecting them to ensure:
# - Coverage across different robot model types where possible
# - Mix of short and long coverage robots in both sets
# - No single model type is completely absent from training

# All unique robots
all_robots = robot_sizes["robot_id"].tolist()

# Hold out every 4th robot from the sorted list for balanced coverage
test_robots  = all_robots[::4][:4]
train_robots = [r for r in all_robots if r not in test_robots]

print(f"\nTraining robots ({len(train_robots)}): {train_robots}")
print(f"Test robots     ({len(test_robots)}):  {test_robots}")

# Validate no overlap
assert len(set(train_robots) & set(test_robots)) == 0, "Overlap detected between train and test robots."
print("\nNo overlap between train and test robots. Split is clean.")

# Create train and test sets 
train_df = model_df[model_df["robot_id"].isin(train_robots)].copy()
test_df  = model_df[model_df["robot_id"].isin(test_robots)].copy()

print(f"\nTrain set : {train_df.shape[0]} rows, {train_df['robot_id'].nunique()} robots")
print(f"Test set  : {test_df.shape[0]} rows, {test_df['robot_id'].nunique()} robots")

# Check model type coverage in both sets
print(f"\nModel types in train : {sorted(train_df['model_type'].unique())}")
print(f"Model types in test  : {sorted(test_df['model_type'].unique())}")

Robot Reading Counts (sorted)


,robot_id,reading_count,model_type
0,ROB-0023,1357,ARM-X9
1,ROB-0027,1353,CONVEYOR-C2
2,ROB-0028,1225,ARM-X7
3,ROB-0018,1225,ARM-X7
4,ROB-0032,1185,ARM-X7
5,ROB-0013,1181,ASSEMBLY-A3
6,ROB-0007,1137,ARM-X7
7,ROB-0037,929,ARM-X9
8,ROB-0049,901,ARM-X7
9,ROB-0042,845,ARM-X9



Training robots (13): ['ROB-0027', 'ROB-0028', 'ROB-0018', 'ROB-0013', 'ROB-0007', 'ROB-0037', 'ROB-0042', 'ROB-0035', 'ROB-0009', 'ROB-0031', 'ROB-0044', 'ROB-0043', 'ROB-0016']
Test robots     (4):  ['ROB-0023', 'ROB-0032', 'ROB-0049', 'ROB-0014']

No overlap between train and test robots. Split is clean.

Train set : 11221 rows, 13 robots
Test set  : 4024 rows, 4 robots

Model types in train : ['ARM-X7', 'ARM-X9', 'ASSEMBLY-A3', 'CONVEYOR-C2']
Model types in test  : ['ARM-X7', 'ARM-X9']
